# SQLAlchemy + PostgreySQL

### SQLAlchemy has four main pieces

1. Engine: the connection to the database
2. Session: a workspace where you add, change and read data, then save
3. Model: a Python class that represents a table
4. Query: built with select(), update(), delete()

Flow: Engine -> Session -> Models -> Queries.

In [ ]:
%pip install sqlalchemy psycopg2-binary python-dotenv pandas alembic

my_project/
├── .env
├── .gitignore
├── db.py
├── models.py
└── main.py

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
from sqlalchemy.engine import URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password="YOUR_PASSWORD",
    host="localhost",
    port=5432,
    database="churn_db",
)
engine = create_engine(url)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM customers"))
    print(result.scalar())

- engine.connect() opens a connection. The with block closes it automatically.
- text() wraps your SQL string. SQLAlchemy 2.0 requires it.
- .scalar() returns the first value of the first row.

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT customerid, state FROM customers LIMIT 3"))

    print(result.fetchall())       # all rows as a list

- .fetchall() ->  all rows
- .fetchone() ->  the next single row
- .scalar()	  ->  one single value
- .mappings().all()	->  rows as dictionaries, like row["state"]

In [ ]:
with engine.connect() as conn:
    rows = conn.execute(text("SELECT customerid, state FROM customers LIMIT 3")).mappings().all()
    for r in rows:
        print(r["customerid"], r["state"])

Put :name in the query and pass values separately. This is safer and cleaner.

In [ ]:
query = text("SELECT * FROM subscriptions WHERE plan_type = :plan AND monthly_charges > :price LIMIT 5")

with engine.connect() as conn:
    rows = conn.execute(query, {"plan": "Premium", "price": 20}).fetchall()
    print(rows)

In SQLAlchemy 2.0, changes are not saved until you commit. There are two ways.

- Way 1: engine.begin() commits automatically at the end of the block (recommended):

In [ ]:
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE IF NOT EXISTS test_table (id INT, note TEXT)"))
    conn.execute(text("INSERT INTO test_table VALUES (:id, :note)"), {"id": 1, "note": "hello"})

Way 2: engine.connect() and commit yourself:

In [ ]:
with engine.connect() as conn:
    conn.execute(text("INSERT INTO test_table VALUES (:id, :note)"), {"id": 2, "note": "world"})
    conn.commit()